# Likelihood Ratio Test (LRT) demo

When fitting models of different sizes (number of free parameters), a larger model usually achieves higher likelihood on the same dataset. The Likelihood Ratio Test helps decide whether that gain is meaningful or mostly model complexity.

For null model $M_0$ with likelihood $L_0$ and alternate model $M_{alt}$ with likelihood $L_{alt}$, the statistic is:

$$LRT = 2(\log L_{alt} - \log L_0).$$

Under regular conditions (Wilks' theorem), this is approximately $\chi^2(d)$ with $d = p_{alt} - p_0$ degrees of freedom, where $p$ is parameter count.

## Demo

This demo now supports an arbitrary number of components for:
- the synthetic data generator,
- the null-hypothesis model ($H_0$), and
- the alternate model ($H_{alt}$).

Data are generated from an equal-weight 1D Gaussian mixture with unit variance per component and means spread across `mean_dist`.

The plots show:
- synthetic data with its generating PDF,
- the fitted null model PDF and components,
- the fitted alternate model PDF and components,
- an LRT summary panel (statistic, df, p-value, and 90/95/99% thresholds).

### Interaction

Row 1 sliders:
- `N_points` — number of sampled points
- `mean_dist` — spread of generating component means
- `N_bins` — histogram bins
- `Resample` — draw a fresh dataset

Row 2 sliders:
- `n_data_comps` — number of components in the true data-generating mixture
- `n_null_model_comps` — number of components in $H_0$
- `n_alt_model_comps` — number of components in $H_{alt}$

For a standard nested-model LRT, choose `n_alt_model_comps > n_null_model_comps`.

### Requirements

Install dependencies if needed:

```
pip install numpy scipy matplotlib scikit-learn ipywidgets jupyterlab_widgets
```

Then run the code cell below.

In [13]:
# Cell 2: Full interactive demo code (generalized component counts)
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
from scipy.stats import chi2, norm
from sklearn.mixture import GaussianMixture
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Utility / helper functions
# -------------------------

def make_component_means(n_components, mean_dist):
    """Evenly spaced component means centered at 0."""
    if n_components <= 1:
        return np.array([0.0])
    return np.linspace(-mean_dist / 2.0, mean_dist / 2.0, n_components)

def sample_true_distribution(N, mean_dist, n_components, rng=None):
    """Sample N points from an equal-weight 1D Gaussian mixture (unit variance per component)."""
    if rng is None:
        rng = np.random.RandomState()
    means = make_component_means(n_components, mean_dist)
    components = rng.choice(np.arange(n_components), size=N, p=np.ones(n_components) / n_components)
    x = rng.normal(loc=means[components], scale=1.0, size=N)
    return x

def sample_random_normal_in_range(N, low, high, rng=None):
    """Sample N points from a normal distribution truncated to [low, high]."""
    if rng is None:
        rng = np.random.RandomState()
    if high <= low:
        return np.full(N, low, dtype=float)

    mu = 0.5 * (low + high)
    sigma = max((high - low) / 6.0, 1e-6)

    samples = np.empty(N, dtype=float)
    filled = 0
    while filled < N:
        draws = rng.normal(loc=mu, scale=sigma, size=max(2 * (N - filled), 64))
        draws = draws[(draws >= low) & (draws <= high)]
        if draws.size == 0:
            continue
        n_take = min(draws.size, N - filled)
        samples[filled:filled + n_take] = draws[:n_take]
        filled += n_take
    return samples

def true_mixture_pdf(xs, mean_dist, n_components):
    """PDF of the synthetic data-generating equal-weight, unit-variance Gaussian mixture."""
    means = make_component_means(n_components, mean_dist)
    pdf = np.zeros_like(xs, dtype=float)
    for m in means:
        pdf += norm.pdf(xs, loc=m, scale=1.0)
    return pdf / n_components

def fit_gmm(X, k, random_state=0, n_init=5):
    """Fit a k-component GaussianMixture, return fitted model and total log-likelihood."""
    gm = GaussianMixture(n_components=k, covariance_type='full', random_state=random_state, n_init=n_init)
    gm.fit(X.reshape(-1, 1))
    ll = gm.score(X.reshape(-1, 1)) * len(X)
    return gm, ll

def gmm_pdf(gm, xs):
    """Return pdf values for a fitted sklearn GMM over xs (1D)."""
    pdf = np.zeros_like(xs, dtype=float)
    weights = gm.weights_
    means = gm.means_.flatten()
    covs = gm.covariances_
    if covs.ndim == 3:
        variances = covs.reshape(covs.shape[0], -1)[:, 0]
    else:
        variances = covs
    stds = np.sqrt(variances)
    for w, m, s in zip(weights, means, stds):
        pdf += w * norm.pdf(xs, loc=m, scale=s)
    return pdf

def gmm_components(gm):
    """Return list of (weight, mean, std) for components of a fitted GMM."""
    weights = gm.weights_
    means = gm.means_.flatten()
    covs = gm.covariances_
    if covs.ndim == 3:
        variances = covs.reshape(covs.shape[0], -1)[:, 0]
    else:
        variances = covs
    stds = np.sqrt(variances)
    return list(zip(weights, means, stds))

def gmm_param_count(k):
    """Count free parameters for a 1D GMM: (k-1) priors + k means + k variances = 3k - 1."""
    return 3 * k - 1

def chi2_thresholds(df):
    return {
        '90%': chi2.ppf(0.90, df),
        '95%': chi2.ppf(0.95, df),
        '99%': chi2.ppf(0.99, df),
    }

# -------------------------
# Persistent state & widgets
# -------------------------

state = {
    'X': None,
    'rng': np.random.RandomState(),
    'is_random_data': False
}

resample_button = widgets.Button(
    description="Resample",
    button_style='primary',
    tooltip="Resample synthetic data"
 )
N_points_slider = widgets.IntSlider(value=1500, min=50, max=5000, step=50, description="N_points", continuous_update=True)
mean_dist_slider = widgets.FloatSlider(value=6.0, min=0.0, max=64.0, step=0.1, description="mean_dist", continuous_update=True)
N_bins_slider = widgets.IntSlider(value=40, min=5, max=200, step=1, description="N_bins", continuous_update=True)

n_data_comps_slider = widgets.IntSlider(value=2, min=1, max=8, step=1, description="n_data_comps", continuous_update=True)
n_null_model_comps_slider = widgets.IntSlider(value=1, min=1, max=8, step=1, description="n_null_model_comps", continuous_update=True)
n_alt_model_comps_slider = widgets.IntSlider(value=2, min=1, max=8, step=1, description="n_alt_model_comps", continuous_update=True)
random_button = widgets.Button(description="Random", tooltip="Generate random normal samples in the current range")

controls_row_1 = widgets.HBox([resample_button, N_points_slider, mean_dist_slider, N_bins_slider])
controls_row_2 = widgets.HBox([n_data_comps_slider, n_null_model_comps_slider, n_alt_model_comps_slider, random_button])
controls = widgets.VBox([controls_row_1, controls_row_2])

out = widgets.Output(layout=widgets.Layout(border='1px solid lightgray'))

# -------------------------
# Update / draw function
# -------------------------

def update(resample=False):
    """Recompute data (optionally resample), fit null/alt models, and redraw the 2x2 grid."""
    n_data = int(n_data_comps_slider.value)
    n_null = int(n_null_model_comps_slider.value)
    n_alt = int(n_alt_model_comps_slider.value)

    if resample or state['X'] is None:
        state['X'] = sample_true_distribution(
            N_points_slider.value, mean_dist_slider.value, n_data, rng=state['rng']
        )
        state['is_random_data'] = False
    X = state['X']

    g_null, ll_null = fit_gmm(X, n_null, random_state=0, n_init=5)
    g_alt, ll_alt = fit_gmm(X, n_alt, random_state=1, n_init=5)

    LRT = 2.0 * (ll_alt - ll_null)
    df = gmm_param_count(n_alt) - gmm_param_count(n_null)

    with out:
        clear_output(wait=True)
        plt.close('all')

        fig, axs = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
        ax1, ax2 = axs[0]
        ax3, ax4 = axs[1]

        xmin, xmax = X.min() - 1.5, X.max() + 1.5
        xs = np.linspace(xmin, xmax, 800)

        # --- Data generating distribution (top-left) ---
        ax1.hist(X, bins=N_bins_slider.value, density=True, alpha=0.85, edgecolor='w', facecolor='k')
        if not state['is_random_data']:
            pdf_true = true_mixture_pdf(xs, mean_dist_slider.value, n_data)
            ax1.plot(xs, pdf_true, '--', lw=2, label=f'True data PDF ({n_data} comps)')
            ax1.set_title('Synthetic data distribution')
        else:
            ax1.set_title('Random normal data distribution')
        ax1.set_ylabel('Density')
        ax1.legend(fontsize=8)

        # --- Null model fit (top-right) ---
        ax2.hist(X, bins=N_bins_slider.value, density=True, alpha=0.85, edgecolor='w', facecolor='k')
        pdf_null = gmm_pdf(g_null, xs)
        ax2.plot(xs, pdf_null, lw=2, label=f'Null PDF ({n_null} comps)')
        for comp_ind, (w, m, s) in enumerate(gmm_components(g_null)):
            ax2.plot(xs, w * norm.pdf(xs, loc=m, scale=s), '--', linewidth=1.5, label=f'null comp {comp_ind}')
        ax2.set_title(f'Null model fit — LL = {ll_null:.2f}')
        ax2.legend(fontsize=7, loc='upper right')

        # --- Alternate model fit (bottom-left) ---
        ax3.hist(X, bins=N_bins_slider.value, density=True, alpha=0.85, edgecolor='w', facecolor='k')
        pdf_alt = gmm_pdf(g_alt, xs)
        ax3.plot(xs, pdf_alt, lw=2, label=f'Alt PDF ({n_alt} comps)')
        for comp_ind, (w, m, s) in enumerate(gmm_components(g_alt)):
            ax3.plot(xs, w * norm.pdf(xs, loc=m, scale=s), '--', linewidth=1.5, label=f'alt comp {comp_ind}')
        ax3.set_title(f'Alternate model fit — LL = {ll_alt:.2f}')
        ax3.set_xlabel('x')
        ax3.set_ylabel('Density')
        ax3.legend(fontsize=7, loc='upper right')

        # --- LRT text (bottom-right) ---
        ax4.axis('off')

        if df > 0:
            p_value = 1.0 - chi2.cdf(LRT, df)
            thr = chi2_thresholds(df)
            lines = [
                f"M_1: {n_null}-component GMM",
                f"M_2: {n_alt}-component GMM",
                "",
                f"log L(M_1) = {ll_null:.3f}",
                f"log L(M_2) = {ll_alt:.3f}",
                f"LRT = 2*(logL(M_2) - logL(M_1)) = {LRT:.3f}",
                f"df = {df}",
                f"p-value (chi-square approx) = {p_value:.6g}",
                "",
                "H_0: M_2 represents no significant improvement over M_1",
                "",
                f"Reject H_0 at 90% if LRT >= {thr['90%']:.3f}",
                f"Reject H_0 at 95% if LRT >= {thr['95%']:.3f}",
                f"Reject H_0 at 99% if LRT >= {thr['99%']:.3f}",
                "",
                "Note: Wilks' theorem is approximate for GMMs.",
                "Interpret p-values as illustrative.",
            ]
        else:
            lines = [
                f"M_1: {n_null}-component GMM",
                f"M_2: {n_alt}-component GMM",
                "",
                f"LRT = {LRT:.3f}, but df = {df}.",
                "Choose n_alt_model_comps > n_null_model_comps",
                "for a standard nested-model LRT.",
            ]

        text = "\n".join(lines)
        ax4.text(
            0.01, 0.99, text,
            transform=ax4.transAxes,
            va='top', ha='left',
            family='monospace', fontsize=9,
            wrap=True
        )

        for ax in [ax1, ax2, ax3]:
            ax.set_xlim(xmin, xmax)

        display(fig)
        plt.close(fig)

# -------------------------
# Event handlers
# -------------------------

def on_resample_clicked(_):
    update(resample=True)

def on_random_clicked(_):
    low = -0.5 * mean_dist_slider.value
    high = 0.5 * mean_dist_slider.value
    if high <= low:
        low, high = -1.0, 1.0
    state['X'] = sample_random_normal_in_range(
        N_points_slider.value, low, high, rng=state['rng']
    )
    state['is_random_data'] = True
    update(resample=False)

def on_data_slider_change(change):
    if change['name'] == 'value':
        update(resample=True)

def on_model_or_view_slider_change(change):
    if change['name'] == 'value':
        update(resample=False)

resample_button.on_click(on_resample_clicked)
random_button.on_click(on_random_clicked)

N_points_slider.observe(on_data_slider_change, names='value')
mean_dist_slider.observe(on_data_slider_change, names='value')
n_data_comps_slider.observe(on_data_slider_change, names='value')

N_bins_slider.observe(on_model_or_view_slider_change, names='value')
n_null_model_comps_slider.observe(on_model_or_view_slider_change, names='value')
n_alt_model_comps_slider.observe(on_model_or_view_slider_change, names='value')

# -------------------------
# Initial draw & display
# -------------------------

update(resample=True)
display(controls, out)

Output(layout=Layout(border_bottom='1px solid lightgray', border_left='1px solid lightgray', border_right='1px…